In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
import warnings

In [2]:
def diff_calculation(SChar, VChar, type, mutate_matrix):
    matrix_values = mutate_matrix.values
    aa_to_idx = {aa: i for i, aa in enumerate(mutate_matrix.columns)}
    GAP = '-'

    difference = []
    for a, b in zip(SChar, VChar):
        if a == GAP or b == GAP or a not in aa_to_idx or b not in aa_to_idx:
            difference.append(0)
            continue

        if type == 'one-hot':
            diff = 0 if a == b else 1
        elif type == 'mut-mat':
            ia, ib = aa_to_idx[a], aa_to_idx[b]
            s_xx = matrix_values[ia, ia]
            s_yy = matrix_values[ib, ib]
            s_xy = matrix_values[ia, ib]
            diff = s_xx + s_yy - 2 * s_xy
        else:
            raise ValueError(f"未知类型: {type}")

        difference.append(diff)

    return difference

def pair_representation(serumHA, virusHA, type, mutate_matrix):
    serumChar = [char for char in serumHA]
    virusChar = [char for char in virusHA]

    if len(serumChar) != len(virusChar):
        return [np.nan for _ in range(len(serumChar))]

    return diff_calculation(serumChar, virusChar, type=type, mutate_matrix=mutate_matrix)

def split_data_by_strain(dataframe, identity_cols, frac):
    strains = dataframe[identity_cols].drop_duplicates().reset_index(drop=True)
    test_strains = strains.sample(frac=frac, random_state=42).reset_index(drop=True)
    test_strains_set = set(zip(*test_strains[identity_cols].values.T))

    mask = dataframe[identity_cols].apply(
        lambda row: tuple(row) in test_strains_set, axis=1)
    test_data = dataframe[mask]
    train_data = dataframe[~mask]

    return train_data, test_data

def rename_fasta_seq(input_file, sequence_data, output_file):
    records = list(SeqIO.parse(input_file, "fasta"))
    new_records = []
    for record, seq in zip(records, sequence_data):
        record.id = str(seq)
        record.description = ""
        new_records.append(record)
    SeqIO.write(new_records, output_file, "fasta")

In [3]:
table_path = '/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/data/new_try_40/data4model(CDC).csv'
save_path = '/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/data/data_40/CDC_all.csv'
origin_table = pd.read_csv(table_path)

## 1. Add sequence ID for embedding & remove duplicates

In [4]:
HA_sequences = pd.concat([origin_table['serumHA'], origin_table['virusHA']]).unique().tolist()
NA_sequences = pd.concat([origin_table['serumNA'], origin_table['virusNA']]).unique().tolist()
HA_dict = {key: value for key, value in zip(HA_sequences, ['HA_' + str(i) for i in range(len(HA_sequences))])}
NA_dict = {key: value for key, value in zip(NA_sequences, ['NA_' + str(i) for i in range(len(NA_sequences))])}

In [5]:
Data_final_1 = origin_table.copy()
Data_final_1.columns = ['serumName', 'serumPassage', 'serumPassCat', 'serumDate', 'Type',
                         'ferret', 'virusName', 'virusPassage', 'virusPassCat', 'virusDate',
                         'virusType', 'dataSource', 'serumIslID', 'serumMatchedPass',
                         'virusIslID', 'virusMatchedPass', 'label', 'seq_a', 'seq_b',
                         'seq_c', 'seq_d']
Data_final_1['seq_id_a'] = Data_final_1['seq_a'].map(HA_dict)
Data_final_1['seq_id_b'] = Data_final_1['seq_b'].map(NA_dict)
Data_final_1['seq_id_c'] = Data_final_1['seq_c'].map(HA_dict)
Data_final_1['seq_id_d'] = Data_final_1['seq_d'].map(NA_dict)

Data_final_1 = Data_final_1[['seq_id_a', 'seq_id_b', 'seq_id_c', 'seq_id_d', 'seq_a', 'seq_b', 'seq_c', 'seq_d',
                               'serumPassCat', 'virusPassCat', 'serumName', 'virusName', 'serumDate', 'virusDate',
                               'Type', 'serumIslID', 'virusIslID', 'label']]

Data_final_2 = Data_final_1.copy()
group_columns = ['seq_a', 'seq_b', 'seq_d', 'seq_c', 'serumPassCat', 'virusPassCat', 'serumName', 'virusName']
agg_dict = {c: 'first' for c in Data_final_2.columns if c not in group_columns}
agg_dict['label'] = 'mean'
Data_final_2 = Data_final_2.groupby(group_columns).agg(agg_dict).reset_index()

In [6]:
Data_final = Data_final_2[['seq_id_a', 'seq_id_b', 'seq_id_c', 'seq_id_d', 'seq_a', 'seq_b', 'seq_c', 'seq_d',
                             'serumPassCat', 'virusPassCat', 'serumName', 'virusName', 'serumDate', 'virusDate',
                             'Type', 'serumIslID', 'virusIslID', 'label']].copy()

## 2. Add columns for Adaboost and Nextflu

In [7]:
mut_mat_path = './GIAG010101.csv'
mutate_matrix = pd.read_csv(mut_mat_path, index_col=0)
# meta feature for model training
meta_features = ['virusName',   # virus avidity (based on both name and passage)
                 'serumName',   # antiserum potency (based on both name and passage)
                 'virusPassCat',   # virus passage category
                 'serumPassCat']   # serum passage category

need_map = True

In [8]:
if need_map:
    H1_dt = Data_final[Data_final['Type'] == 'H1N1']
    H1_sequence = pd.concat([H1_dt['seq_a'], H1_dt['seq_c']], axis=0).reset_index(drop=True).drop_duplicates().reset_index(drop=True)
    H1_seq = [SeqRecord(Seq(seq), id=str(seq), description="") for seq in H1_sequence]
    output_file = "./H1_input.fasta"
    SeqIO.write(H1_seq, output_file, "fasta")

    H3_dt = Data_final[Data_final['Type'] == 'H3N2']
    H3_sequence = pd.concat([H3_dt['seq_a'], H3_dt['seq_c']], axis=0).reset_index(drop=True).drop_duplicates().reset_index(drop=True)
    H3_seq = [SeqRecord(Seq(seq), id=str(seq), description="") for seq in H3_sequence]
    output_file = "./H3_input.fasta"
    SeqIO.write(H3_seq, output_file, "fasta")

    # mafft --thread 12 --auto --inputorder "H1_input.fasta" > "H1_output.fasta"
    # mafft --thread 12 --auto --inputorder "H3_input.fasta" > "H3_output.fasta"

In [9]:
if need_map:
    rename_fasta_seq('H1_output.fasta', H1_sequence,'H1_output.fasta')
    rename_fasta_seq('H3_output.fasta', H3_sequence,'H3_output.fasta')

In [10]:
# 1. add mapped columns
H1_dict = {rec.id: str(rec.seq) for rec in SeqIO.parse("H1_output.fasta", "fasta")}
H3_dict = {rec.id: str(rec.seq) for rec in SeqIO.parse("H3_output.fasta", "fasta")}
merged_dict = H1_dict | H3_dict
Data_final['serumHA'] = Data_final['seq_a'].map(merged_dict)
Data_final['virusHA'] = Data_final['seq_c'].map(merged_dict)
missing_a = Data_final.loc[Data_final['serumHA'].isna(), 'seq_a'].unique()
missing_c = Data_final.loc[Data_final['virusHA'].isna(), 'seq_c'].unique()
if len(missing_a) or len(missing_c):
    warnings.warn(
        "以下 seq_id 在 FASTA 中未找到，映射为 NaN: \n"
        f"  serumHA 缺失: {list(missing_a)}\n"
        f"  virusHA 缺失: {list(missing_c)}"
    )

# 2. remove space in Name
Data_final['serumName'] = Data_final['serumName'].str.replace(
    ' ', '', regex=False)
Data_final['virusName'] = Data_final['virusName'].str.replace(
    ' ', '', regex=False)

# 3. calculate sequence difference
diff_mat_list = []
diff_ohe_list = []
for ref_seq, test_seq in tqdm(zip(Data_final['serumHA'], Data_final['virusHA']), total=len(Data_final), desc="Processing sequences"):
    if not ref_seq or not test_seq or pd.isna(ref_seq) or pd.isna(test_seq):
        diff_mat_list.append(np.nan)
        diff_ohe_list.append(np.nan)
        continue

    diff_mat_list.append(pair_representation(
        ref_seq, test_seq, type='mut-mat', mutate_matrix=mutate_matrix))
    diff_ohe_list.append(pair_representation(
        ref_seq, test_seq, type='one-hot', mutate_matrix=mutate_matrix))

Data_final["seq_diff_mat"] = diff_mat_list
Data_final["seq_diff_ohe"] = diff_ohe_list

Processing sequences: 100%|██████████| 34312/34312 [00:17<00:00, 1921.78it/s]


In [12]:
Data_final.to_csv(save_path)